**Author:** Adebanji Oluwatimileyin Adelowo  
**GitHub:** [adebanjiadelowo](https://github.com/adebanjiadelowo)

<div align="center">
<h1>Enterprise NL2SQL Pipeline</h1>
<h2>Spider Benchmark Evaluation</h2>
<p>Replaces hand-crafted test questions with real Q&amp;A pairs from the Spider NL2SQL benchmark.</p>
</div>

<hr>

## Overview

The hand-crafted notebooks use manually written questions with no ground-truth SQL to compare against.
This notebook swaps those in for **`b-mc2/sql-create-context`** — a HuggingFace dataset derived from
the Spider and WikiSQL benchmarks — giving us:

- Real natural-language questions written by human annotators
- Gold SQL to compare our output against
- Multi-table schemas that make the table-selector non-trivial

**Evaluation metrics:**
| Metric | What it measures |
|---|---|
| Table selection accuracy | Did the model pick the right tables? |
| SQL exact match | Does generated SQL equal gold SQL (normalised)? |

> **Prerequisite:** `OPENAI_API_KEY` in a `.env` file.

In [ ]:
!pip install -q openai python-dotenv datasets pandas

In [ ]:
import os, re, json
import pandas as pd
from openai import OpenAI
from datasets import load_dataset
from dotenv import load_dotenv

load_dotenv(dotenv_path="../../../.env")
load_dotenv(dotenv_path="../../.env")
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-4o-mini"


## 1 – Load the dataset

In [ ]:
ds = load_dataset("b-mc2/sql-create-context", split="train")
print(f"Total examples : {len(ds):,}")

sample = ds[100]
print("\nSample question :", sample["question"])
print("Sample answer   :", sample["answer"])
print("Sample context  :", sample["context"][:300], "...")


## 2 – Filter for multi-table examples

Single-table examples are trivial for the table selector. We keep only examples where the schema contains **3 or more** `CREATE TABLE` blocks.


In [ ]:
def count_tables(context: str) -> int:
    return len(re.findall(r"CREATE TABLE", context, re.IGNORECASE))

multi = ds.filter(lambda x: count_tables(x["context"]) >= 3)
print(f"Multi-table examples (>=3 tables): {len(multi):,}")
print(f"Fraction of total                : {len(multi)/len(ds):.1%}")

from collections import Counter
dist = Counter(count_tables(x["context"]) for x in ds)
print("\nTable count distribution:")
for k in sorted(dist):
    print(f"  {k} tables: {dist[k]:,} examples")


## 3 – Schema parsing helpers

In [ ]:
def parse_table_names(context: str) -> list[str]:
    return re.findall(r'CREATE TABLE\s+[`"]?(\w+)[`"]?', context, re.IGNORECASE)


def parse_columns(context: str, table: str) -> list[str]:
    pattern = r'CREATE TABLE\s+[`"]?' + re.escape(table) + r'[`"]?\s*\(([^;]+)'
    m = re.search(pattern, context, re.IGNORECASE | re.DOTALL)
    if not m:
        return []
    skip = {"PRIMARY", "FOREIGN", "UNIQUE", "CHECK", "CONSTRAINT", "KEY", "INDEX"}
    cols = []
    for line in m.group(1).split(","):
        line = line.strip()
        if not line:
            continue
        word = line.split()[0].strip('`"') if line.split() else ""
        if word.upper() not in skip and word:
            cols.append(word)
    return cols


def build_catalogue(context: str) -> dict[str, str]:
    catalogue = {}
    for t in parse_table_names(context):
        cols = parse_columns(context, t)
        cols_str = ", ".join(cols[:6])
        desc = f"{t}: columns — {cols_str}"
        if len(cols) > 6:
            desc += f" (+{len(cols)-6} more)"
        catalogue[t] = desc
    return catalogue


def extract_gold_tables(sql: str) -> set[str]:
    tables = set()
    for m in re.finditer(r'\b(?:FROM|JOIN)\s+[`"]?(\w+)[`"]?', sql, re.IGNORECASE):
        tables.add(m.group(1).lower())
    return tables


# Smoke test
ex = multi[0]
cat = build_catalogue(ex["context"])
print("Catalogue for first example:")
for t, d in cat.items():
    print(f"  {d}")
print("\nGold tables:", extract_gold_tables(ex["answer"]))


## 4 – Pipeline functions

In [ ]:
SYSTEM_SELECT = (
    "You are a database assistant. "
    "Given a list of database tables and their descriptions, identify which tables are required "
    "to answer the user's question. "
    "Respond ONLY with a valid JSON array of table name strings. "
    "Do not include any explanation or markdown fencing."
)

USER_SELECT = """
### Available tables
{tables}

### Question
{question}

Which tables are needed? Reply with a JSON array of table names only.
"""


def select_tables(question: str, catalogue: dict[str, str]) -> list[str]:
    tables_text = "\n".join(f"{t}: {d}" for t, d in catalogue.items())
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_SELECT},
            {"role": "user",   "content": USER_SELECT.format(tables=tables_text, question=question)},
        ],
        temperature=0,
    )
    selected = json.loads(response.choices[0].message.content.strip())
    known = {k.lower() for k in catalogue}
    return [t for t in selected if t.lower() in known]


def strip_fencing(text: str) -> str:
    text = re.sub(r"^```(?:sql)?\s*", "", text.strip(), flags=re.IGNORECASE)
    return re.sub(r"\s*```$", "", text.strip()).strip()


def generate_sql(question: str, context: str, selected: list[str]) -> str:
    tbl_pattern = r'CREATE TABLE\s+[`"]?(' + "|".join(re.escape(t) for t in selected) + r')[`"]?'
    blocks = []
    for block in re.split(r"(?=CREATE TABLE)", context, flags=re.IGNORECASE):
        if selected and re.search(tbl_pattern, block, re.IGNORECASE):
            blocks.append(block.strip())
    schema = "\n\n".join(blocks) if blocks else context
    prompt = (
        schema + "\n\n"
        "-- Write valid SQL. Use only the tables listed above.\n"
        "-- Return ONLY the raw SQL query, no explanation, no markdown fencing.\n\n"
        "Question: " + question
    )
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are an expert SQL engineer. Return only valid SQL, no markdown."},
            {"role": "user",   "content": prompt},
        ],
        temperature=0,
    )
    return strip_fencing(response.choices[0].message.content)


## 5 – Evaluation run

For each of the first `N` multi-table examples:
1. Build a catalogue from the schema context
2. Run the table selector
3. Generate SQL from selected schemas only
4. Compare selected tables to tables used in the gold SQL
5. Compare generated SQL to gold SQL (normalised)


In [ ]:
def normalize_sql(sql: str) -> str:
    return re.sub(r"\s+", " ", sql.lower().strip().rstrip(";"))


def run_evaluation(dataset, n: int = 30) -> pd.DataFrame:
    rows = []
    for ex in dataset.select(range(n)):
        q        = ex["question"]
        ctx      = ex["context"]
        gold_sql = ex["answer"]
        gold_tbl = extract_gold_tables(gold_sql)
        catalogue = build_catalogue(ctx)
        try:
            selected = select_tables(q, catalogue)
            sql      = generate_sql(q, ctx, selected)
        except Exception as e:
            selected, sql = [], f"ERROR: {e}"
        selected_lower = {t.lower() for t in selected}
        table_ok = selected_lower == gold_tbl
        sql_ok   = normalize_sql(sql) == normalize_sql(gold_sql)
        rows.append({
            "question":        q,
            "gold_tables":     sorted(gold_tbl),
            "selected_tables": sorted(selected_lower),
            "table_match":     table_ok,
            "gold_sql":        gold_sql,
            "generated_sql":   sql,
            "sql_exact_match": sql_ok,
        })
        print(f"  [{len(rows):2d}/{n}] table={table_ok!s:<5}  sql={sql_ok!s:<5}  Q: {q[:55]}")
    return pd.DataFrame(rows)


print("Running evaluation on 30 examples ...")
df = run_evaluation(multi, n=30)


## 6 – Results

In [ ]:
table_acc = df["table_match"].mean()
sql_acc   = df["sql_exact_match"].mean()

print("=" * 52)
print(f"Table selection accuracy : {table_acc:.1%}  ({int(df['table_match'].sum())}/{len(df)})")
print(f"SQL exact match          : {sql_acc:.1%}  ({int(df['sql_exact_match'].sum())}/{len(df)})")
print("=" * 52)
print()
print("Note: SQL exact match is strict (string equality after normalisation).\n"
      "Execution match would give a higher and fairer score but requires the actual DB files.")


In [ ]:
failures = df[~df["table_match"]][["question", "gold_tables", "selected_tables"]].reset_index(drop=True)
print(f"Table selection mismatches ({len(failures)}):")
print(failures.to_string(index=False))


In [ ]:
df[["question", "table_match", "sql_exact_match"]]

## Conclusions

Using `b-mc2/sql-create-context` instead of hand-crafted questions gives a
**reproducible, quantitative baseline** for both pipeline stages.

**Key observations:**
- Table selection accuracy shows how well column-name-based descriptions guide the model
- SQL exact match is a lower bound — semantically equivalent queries may differ in syntax
- Table selection failures cascade into SQL failures, confirming Stage 1 is the critical bottleneck

**Ways to improve scores:**
- Write richer table descriptions (not just column names)
- Add few-shot table→question examples in the selector prompt
- Use execution match instead of exact match for fairer SQL evaluation
